# Manuscript Figure 3

This is a public Colab adaptation of `src/05_mapping/04_predict_multi_waterbody_water_quality_maps_paper.ipynb` from the team snapshot. It uses the original figure and inference cells with paths set to the archived data. Fitted models and cell outputs are not published. Start with a fresh CPU runtime and run from the top.


## Load the team data

Open the authorized Drive folder and add the shared ZIP to **My Drive** as a shortcut, or upload your copy there. Set `ARCHIVE_PATH` if the file is in a subfolder. This cell verifies and unpacks the exact team snapshot to temporary Colab storage. No Earth Engine login is used. Save generated results to Drive before disconnecting.


In [ ]:
import hashlib
import os
import tempfile
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ARCHIVE_PATH = Path("/content/drive/MyDrive/Water_Quality_Processed_Handoff.zip")
EXPECTED_SHA256 = "1f1039b7a6d113685c2c95f840f2a475be5e936e02090bcf5283f2ba9b506773"
HANDOFF_ROOT = Path("/content/wqr-handoff-" + EXPECTED_SHA256[:12])
marker = HANDOFF_ROOT / ".verified-archive-sha256"

if not (marker.is_file() and marker.read_text().strip() == EXPECTED_SHA256):
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            f"{ARCHIVE_PATH} was not found. Ask the project owner for the team ZIP "
            "and point ARCHIVE_PATH to its location on the mounted Drive."
        )
    with tempfile.TemporaryDirectory(dir=HANDOFF_ROOT.parent) as tmp:
        tmp = Path(tmp)
        local_zip = tmp / ARCHIVE_PATH.name
        digest = hashlib.sha256()
        with ARCHIVE_PATH.open("rb") as src, local_zip.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
                dst.write(chunk)
        if digest.hexdigest() != EXPECTED_SHA256:
            raise ValueError("This ZIP differs from the verified snapshot. Check the filename and download.")
        extracted = tmp / "snapshot"
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(extracted)
        (extracted / marker.name).write_text(EXPECTED_SHA256 + "\n")
        os.replace(extracted, HANDOFF_ROOT)

print("Team data ready:", HANDOFF_ROOT)


# Multi-Waterbody Water-Quality Prediction Maps

This notebook loads the locked model artifacts and applies them to the 62-band moving-window GeoTIFFs produced by Notebook 03. It predicts Turbidity, suspended solids (SS), and dissolved oxygen (DO) without refitting, tuning, or selecting any model.

Primary reporting layers use **S2 + Context**. The S2-only Turbidity and SS layers are retained as diagnostic ablations. DO is an indirect proxy prediction and must not be interpreted as a direct satellite measurement.

## 1. Imports and configuration

All prediction logic is contained in this notebook. No separate runtime Python file is imported.

In [ ]:
# Google Colab setup: mount Drive before importing the model stack.
from pathlib import Path

import importlib.metadata
import importlib.util
import subprocess
import sys

required_packages = {
    "scikit-learn": "1.7.2",
    "catboost": "1.2.10",
}
install_specs = []
for package, version in required_packages.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install_specs.append(f"{package}=={version}")

for module, package in {
    "rasterio": "rasterio",
    "affine": "affine",
    "joblib": "joblib",
    "geopandas": "geopandas",
    "contextily": "contextily",
    "folium": "folium",
}.items():
    if importlib.util.find_spec(module) is None:
        install_specs.append(package)

if install_specs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install_specs])

# Install Noto Latin and Thai fonts for mixed-language map titles.
NOTO_THAI_FONT_PATH = Path("/usr/share/fonts/truetype/noto/NotoSansThai-Regular.ttf")
if not NOTO_THAI_FONT_PATH.exists():
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-noto-core"], check=True)

import base64
import hashlib
import json
import os
import re
import shutil
import warnings

import contextily as ctx
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, TwoSlopeNorm
from matplotlib import font_manager
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from IPython.display import display
from rasterio.features import geometry_mask
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from shapely.geometry import mapping

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Colab runtime ready")
print("scikit-learn:", importlib.metadata.version("scikit-learn"))
print("catboost:", importlib.metadata.version("catboost"))
print("rasterio:", rasterio.__version__)
print("joblib:", joblib.__version__)


In [ ]:
METADATA_ROOT = HANDOFF_ROOT / "data_ready/maps/polygon_metadata"
FEATURE_INPUT_ROOT = HANDOFF_ROOT / "data_ready/rasters/polygon/features"
RGB_INPUT_ROOT = HANDOFF_ROOT / "data_ready/rasters/polygon/rgb"
OUTPUT_ROOT_OVERRIDE = HANDOFF_ROOT / "run_output/colab_fig03"

MODEL_ROOT = OUTPUT_ROOT_OVERRIDE / "locked_models"
OUTPUT_ROOT = OUTPUT_ROOT_OVERRIDE

GEOTIFF_DIR = OUTPUT_ROOT / "geotiff"
TEMP_DIR = OUTPUT_ROOT / "temporary_tiles"
TABLE_DIR = OUTPUT_ROOT / "tables"
QUICKLOOK_DIR = OUTPUT_ROOT / "quicklooks"
for directory in [GEOTIFF_DIR, TEMP_DIR, TABLE_DIR, QUICKLOOK_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = METADATA_ROOT / "mapping_manifest.csv"
BAND_SCHEMA_PATH = METADATA_ROOT / "moving_window_band_schema.csv"
READINESS_PATH = METADATA_ROOT / "notebook03_readiness.csv"
RANGE_AUDIT_PATH = METADATA_ROOT / "map_feature_range_audit.csv"
POLYGON_PATH = HANDOFF_ROOT / "data_ready/maps/polygons/waterbody_polygons.geojson"
REGISTRY_PATH = MODEL_ROOT / "model_registry.json"
FEATURE_MANIFEST_PATH = MODEL_ROOT / "contracts" / "feature_manifest.csv"
TRAINING_RANGE_PATH = MODEL_ROOT / "contracts" / "training_feature_ranges.csv"

# Use ["WB01"] for a smoke test, then None for all seven water bodies.
MAP_IDS_TO_RUN = ["WB02", "WB07", "WB05"]
REQUIRE_SENTINEL2_RGB = True
OVERWRITE_OUTPUTS = False
KEEP_TEMPORARY_TILES = False
BLOCK_SIZE = 512
NODATA = -9999.0
EXPORT_PIXEL_SIZE_M = 20.0
BASEMAP_SOURCE = ctx.providers.Esri.WorldImagery
BASEMAP_ZOOM = "auto"
PREDICTION_VIEW_PADDING_FRACTION = 0.12
PREDICTION_VIEW_PADDING_BY_MAP_ID = {"WB03": 0.02}
PREDICTION_VIEW_MIN_PADDING_PIXELS = 2
MIN_QUICKLOOK_SUPPORT_PIXELS = 1
DISPLAY_OVERVIEW_FOR_MULTIPOLYGON = False

print("Metadata root:", METADATA_ROOT)
print("Feature root :", FEATURE_INPUT_ROOT)
print("RGB root     :", RGB_INPUT_ROOT)
print("Output root  :", OUTPUT_ROOT)


In [ ]:
# Load the exact historical setup cell from the verified team snapshot.
import hashlib
import json

original = HANDOFF_ROOT / "src/05_mapping/04_predict_multi_waterbody_water_quality_maps_paper.ipynb"
private_notebook = json.loads(original.read_text(encoding="utf-8-sig"))
private_setup = "".join(private_notebook["cells"][4]["source"])
if hashlib.sha256(private_setup.encode()).hexdigest() != "f15362b88d0d7788d91cc1f840174a1a0f527b45f23e9ce0bfe1bdb92ec6ffc6":
    raise ValueError("The locked-model setup cell does not match the team snapshot.")
exec(compile(private_setup, str(original), "exec"))
del private_setup, private_notebook


## 2. Input and model completion gates

The notebook stops before prediction if Notebook 03 is incomplete, an artifact checksum changes, a GeoTIFF does not have 62 bands, or a required model feature is absent.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for required_path in [
    MANIFEST_PATH, BAND_SCHEMA_PATH, READINESS_PATH, RANGE_AUDIT_PATH, POLYGON_PATH,
    REGISTRY_PATH, FEATURE_MANIFEST_PATH, TRAINING_RANGE_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

readiness03 = pd.read_csv(READINESS_PATH)
readiness03["passed_bool"] = (
    readiness03["passed"].astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
)
if readiness03["passed_bool"].isna().any():
    raise ValueError("Could not parse some Notebook 03 readiness values.")

BLOCKING_NOTEBOOK03_CHECKS = [
    "reviewed_polygon_complete",
    "same_62_band_schema",
    "training_range_audit_available",
    "earth_engine_exports_completed",
    "feature_geotiffs_valid",
    "sentinel2_rgb_geotiffs_valid",
]
NONBLOCKING_NOTEBOOK03_CHECKS = [
    "all_station_rows_retained",
    "every_map_has_pixel_QA_sample",
]

missing_checks = set(BLOCKING_NOTEBOOK03_CHECKS) - set(readiness03["check"])
if missing_checks:
    raise KeyError(f"Missing blocking readiness checks: {sorted(missing_checks)}")
blocking_rows = readiness03.loc[
    readiness03["check"].isin(BLOCKING_NOTEBOOK03_CHECKS)
].copy()
failed_blocking = blocking_rows.loc[~blocking_rows["passed_bool"]]
display(readiness03[["check", "passed_bool", "detail"]])
# if not failed_blocking.empty:
#     raise RuntimeError(
#         "Critical Notebook 03 checks failed:\n"
#         + failed_blocking[["check", "detail"]].to_string(index=False)
#     )

station_warnings = readiness03.loc[
    readiness03["check"].isin(NONBLOCKING_NOTEBOOK03_CHECKS)
    & ~readiness03["passed_bool"]
]
if not station_warnings.empty:
    print("WARNING: station-point QA is incomplete; map prediction will use only prediction_support pixels.")
    display(station_warnings[["check", "detail"]])

NOTEBOOK03_CRITICAL_READY = True

manifest = pd.read_csv(MANIFEST_PATH)
manifest["map_id"] = manifest["map_id"].astype(str)
band_schema = pd.read_csv(BAND_SCHEMA_PATH).sort_values("band_order")
band_names = band_schema["band"].astype(str).tolist()
if len(band_names) != 62 or len(set(band_names)) != 62:
    raise ValueError("Expected exactly 62 unique bands from Notebook 03.")

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
feature_manifest = pd.read_csv(FEATURE_MANIFEST_PATH)
training_ranges = pd.read_csv(TRAINING_RANGE_PATH)

print("Critical Notebook 03 gate: PASS")
print("Water bodies:", len(manifest))
print("Raster bands:", len(band_names))
print("Locked artifacts:", len(registry["artifacts"]))

waterbody_polygons = gpd.read_file(POLYGON_PATH)
waterbody_polygons["map_id"] = waterbody_polygons["map_id"].astype(str)
if waterbody_polygons.crs is None:
    waterbody_polygons = waterbody_polygons.set_crs("EPSG:4326")
if waterbody_polygons["map_id"].duplicated().any():
    raise ValueError("Expected one Polygon/MultiPolygon feature per map_id.")
missing_polygon_ids = set(manifest["map_id"]) - set(waterbody_polygons["map_id"])
if missing_polygon_ids:
    raise ValueError(f"Missing polygons for: {sorted(missing_polygon_ids)}")
print("Polygon features:", len(waterbody_polygons))

In [ ]:
# Discover only the feature/RGB files matching each manifest map_id and date.
def parse_export_identity(path, token):
    match = re.match(rf"^(WB\d{{2}})_(\d{{8}})_{token}", path.name)
    if match is None:
        return None
    return match.group(1), match.group(2)


expected_dates = {
    str(row.map_id): pd.Timestamp(row.sat_date).strftime("%Y%m%d")
    for row in manifest.itertuples(index=False)
}

inventory_columns = [
    "map_id", "sat_date", "file", "file_name", "band_count", "width", "height", "crs",
    "x_resolution", "y_resolution", "nodata", "valid",
]
inventory_rows = []
for path in sorted(FEATURE_INPUT_ROOT.rglob("*.tif")):
    identity = parse_export_identity(path, "moving_window_features")
    if identity is None:
        continue
    map_id, date_text = identity
    if expected_dates.get(map_id) != date_text:
        continue
    with rasterio.open(path) as src:
        inventory_rows.append({
            "map_id": map_id, "sat_date": date_text, "file": str(path), "file_name": path.name,
            "band_count": src.count, "width": src.width, "height": src.height,
            "crs": str(src.crs), "x_resolution": abs(src.transform.a),
            "y_resolution": abs(src.transform.e), "nodata": src.nodata,
            "valid": src.count == len(band_names) and src.crs is not None,
        })
raster_inventory = pd.DataFrame(inventory_rows, columns=inventory_columns)
if raster_inventory.empty:
    raise FileNotFoundError(
        f"No manifest-date moving_window_features GeoTIFFs below {FEATURE_INPUT_ROOT}. "
        f"Expected: {expected_dates}"
    )
raster_inventory = raster_inventory.sort_values(["map_id", "file_name"]).reset_index(drop=True)

if MAP_IDS_TO_RUN is not None:
    selected_ids = set(MAP_IDS_TO_RUN)
    raster_inventory = raster_inventory.loc[raster_inventory["map_id"].isin(selected_ids)].copy()
    manifest = manifest.loc[manifest["map_id"].isin(selected_ids)].copy()
if manifest.empty:
    raise ValueError("MAP_IDS_TO_RUN did not match the mapping manifest.")
if not raster_inventory["valid"].all():
    raise RuntimeError("At least one feature GeoTIFF failed the 62-band/CRS check.")

expected_ids = set(manifest["map_id"])
found_ids = set(raster_inventory["map_id"])
if expected_ids != found_ids:
    raise RuntimeError(
        f"Manifest/feature IDs differ: missing={expected_ids-found_ids}, unexpected={found_ids-expected_ids}"
    )

rgb_columns = ["map_id", "sat_date", "file", "file_name", "band_count", "crs", "valid"]
rgb_rows = []
for path in sorted(RGB_INPUT_ROOT.rglob("*.tif")):
    identity = parse_export_identity(path, "sentinel2_rgb")
    if identity is None:
        continue
    map_id, date_text = identity
    if expected_dates.get(map_id) != date_text:
        continue
    with rasterio.open(path) as src:
        rgb_rows.append({
            "map_id": map_id, "sat_date": date_text, "file": str(path), "file_name": path.name,
            "band_count": src.count, "crs": str(src.crs),
            "valid": src.count == 3 and src.crs is not None,
        })
rgb_inventory = pd.DataFrame(rgb_rows, columns=rgb_columns)
if MAP_IDS_TO_RUN is not None:
    rgb_inventory = rgb_inventory.loc[rgb_inventory["map_id"].isin(set(MAP_IDS_TO_RUN))].copy()

rgb_ids = set(rgb_inventory["map_id"])
missing_rgb_ids = sorted(set(manifest["map_id"]) - rgb_ids)
if REQUIRE_SENTINEL2_RGB and missing_rgb_ids:
    raise FileNotFoundError(f"Missing manifest-date Sentinel-2 RGB exports for: {missing_rgb_ids}")
if not rgb_inventory.empty and not rgb_inventory["valid"].all():
    raise RuntimeError("At least one RGB GeoTIFF failed the 3-band/CRS check.")

display(raster_inventory)
display(rgb_inventory)
raster_inventory.to_csv(TABLE_DIR / "raster_inventory.csv", index=False, encoding="utf-8-sig")
rgb_inventory.to_csv(TABLE_DIR / "sentinel2_rgb_inventory.csv", index=False, encoding="utf-8-sig")
print("Feature maps ready:", sorted(found_ids))
print("RGB maps ready    :", sorted(rgb_ids))

In [ ]:
# Load the five model bundles directly from their joblib artifacts.
bundles = {}
model_rows = []
for item in registry["artifacts"]:
    artifact_path = MODEL_ROOT / item["artifact_relative_path"].replace("\\", "/")
    if not artifact_path.exists():
        raise FileNotFoundError(artifact_path)
    checksum_ok = sha256_file(artifact_path) == item["artifact_sha256"]
    if not checksum_ok:
        raise RuntimeError(f"Artifact checksum changed: {artifact_path}")
    bundle = joblib.load(artifact_path)
    if bundle["artifact_id"] != item["artifact_id"]:
        raise ValueError(f"Artifact ID mismatch: {artifact_path}")
    missing = [name for name in bundle["feature_names"] if name not in band_names]
    if missing:
        raise KeyError(f"{bundle['artifact_id']} is missing raster features: {missing}")
    bundles[bundle["artifact_id"]] = bundle
    model_rows.append({
        "artifact_id": bundle["artifact_id"],
        "target": bundle["target"],
        "feature_set": bundle["feature_set"],
        "n_features": len(bundle["feature_names"]),
        "prediction_strategy": bundle["prediction_contract"]["strategy"],
        "checksum_ok": checksum_ok,
    })

EXPECTED_ARTIFACT_IDS = [
    "turbidity_s2", "turbidity_s2_context",
    "ss_s2", "ss_s2_context", "do_s2_context",
]
if set(bundles) != set(EXPECTED_ARTIFACT_IDS):
    raise RuntimeError(f"Expected artifacts {EXPECTED_ARTIFACT_IDS}, found {sorted(bundles)}")

model_inventory = pd.DataFrame(model_rows)
display(model_inventory)
model_inventory.to_csv(TABLE_DIR / "model_inventory.csv", index=False, encoding="utf-8-sig")

## 3. Embedded prediction and applicability functions

Turbidity models predict `log1p(Turbidity)` and are transformed back with `expm1`. SS uses the locked probability-weighted two-stage formula. DO uses the raw CatBoost prediction and is reported as an indirect proxy.

In [ ]:
def predict_bundle(bundle, feature_frame):
    """Predict using the contract stored inside one artifact bundle."""
    feature_names = bundle["feature_names"]
    missing = [name for name in feature_names if name not in feature_frame.columns]
    if missing:
        raise KeyError(f"Missing features for {bundle['artifact_id']}: {missing}")
    x = feature_frame.loc[:, feature_names]
    strategy = bundle["prediction_contract"]["strategy"]

    if strategy == "log1p_regression":
        pred_log = np.asarray(bundle["estimator"].predict(x), dtype=float).reshape(-1)
        return np.expm1(np.clip(pred_log, 0, None)), None

    if strategy == "raw_regression":
        prediction = np.asarray(bundle["estimator"].predict(x), dtype=float).reshape(-1)
        return prediction, None

    if strategy == "two_stage_soft_gate":
        p_high = np.asarray(bundle["classifier"].predict_proba(x), dtype=float)[:, 1]
        high_log = np.asarray(bundle["regressor"].predict(x), dtype=float).reshape(-1)
        high_value = np.expm1(np.clip(high_log, 0, None))
        prediction = p_high * high_value + (1.0 - p_high) * float(bundle["low_value"])
        return prediction, p_high

    raise ValueError(f"Unknown prediction strategy: {strategy}")


# Prefer the P01-P99 training bounds recorded by Notebook 03. Fall back to artifact min-max ranges.
range_audit = pd.read_csv(RANGE_AUDIT_PATH)
robust_bounds = (
    range_audit[["feature", "training_p01", "training_p99"]]
    .dropna()
    .groupby("feature", as_index=False)
    .first()
    .set_index("feature")
)
fallback_bounds = (
    training_ranges.groupby("feature_name", as_index=False)
    .agg(training_min=("training_min", "min"), training_max=("training_max", "max"))
    .set_index("feature_name")
)


def feature_bounds(feature_name):
    if feature_name in robust_bounds.index:
        row = robust_bounds.loc[feature_name]
        return float(row["training_p01"]), float(row["training_p99"]), "p01_p99"
    if feature_name in fallback_bounds.index:
        row = fallback_bounds.loc[feature_name]
        return float(row["training_min"]), float(row["training_max"]), "min_max"
    return None


def ood_fraction(feature_frame, feature_names):
    outside = np.zeros(len(feature_frame), dtype=np.float32)
    compared = np.zeros(len(feature_frame), dtype=np.float32)
    for name in feature_names:
        bounds = feature_bounds(name)
        if bounds is None:
            continue
        lower, upper, _ = bounds
        values = pd.to_numeric(feature_frame[name], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(values)
        compared[finite] += 1
        outside[finite] += ((values[finite] < lower) | (values[finite] > upper)).astype(np.float32)
    return np.divide(outside, compared, out=np.full_like(outside, np.nan), where=compared > 0)


def iter_windows(width, height, block_size=512):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=min(block_size, width - col_off),
                height=min(block_size, height - row_off),
            )

In [ ]:
# Output band contract. One analytical GeoTIFF stores every locked prediction and its QA layers.
PREDICTION_BANDS = [
    "turbidity_s2",
    "turbidity_s2_context",
    "ss_s2",
    "ss_s2_context",
    "do_s2_context",
]
PROBABILITY_BANDS = ["ss_s2_p_high", "ss_s2_context_p_high"]
SUPPORT_BAND = "prediction_support"
OOD_BANDS = [f"{artifact_id}_ood_fraction" for artifact_id in PREDICTION_BANDS]
CONTEXT_ARTIFACT_IDS = ["turbidity_s2_context", "ss_s2_context", "do_s2_context"]
CONTEXT_MISSING_BANDS = [f"{artifact_id}_context_missing_fraction" for artifact_id in CONTEXT_ARTIFACT_IDS]
OUTPUT_BANDS = (
    PREDICTION_BANDS + PROBABILITY_BANDS + [SUPPORT_BAND]
    + OOD_BANDS + CONTEXT_MISSING_BANDS
)
OUTPUT_BAND_INDEX = {name: index + 1 for index, name in enumerate(OUTPUT_BANDS)}

source_band_index = {name: index + 1 for index, name in enumerate(band_names)}
required_source_features = sorted({
    feature
    for bundle in bundles.values()
    for feature in bundle["feature_names"]
} | {SUPPORT_BAND})
required_source_indexes = [source_band_index[name] for name in required_source_features]
SPECTRAL_FEATURE_NAMES = set(band_names[:27])

display(pd.DataFrame({
    "output_band": range(1, len(OUTPUT_BANDS) + 1),
    "name": OUTPUT_BANDS,
}))
print("Input bands read per block:", len(required_source_features))
print("Output bands:", len(OUTPUT_BANDS))

## 4. Block-wise prediction

Only pixels with `prediction_support > 0` and complete spectral features are predicted. Missing context values are passed as NaN to the fitted imputers stored inside the locked pipelines. A separate context-missing layer makes this fallback explicit. Unsupported pixels remain nodata. Each source tile is processed independently before tiles belonging to the same `map_id` are mosaicked.

In [ ]:
def process_source_tile(source_path, output_path):
    source_path = Path(source_path)
    output_path = Path(output_path)
    if output_path.exists() and not OVERWRITE_OUTPUTS:
        return output_path
    partial_path = output_path.with_name(f"{output_path.stem}.partial{output_path.suffix}")
    partial_path.unlink(missing_ok=True)

    with rasterio.open(source_path) as src:
        if src.count != len(band_names):
            raise ValueError(f"Band count mismatch in {source_path}")
        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            count=len(OUTPUT_BANDS),
            dtype="float32",
            nodata=NODATA,
            compress="DEFLATE",
            predictor=2,
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(partial_path, "w", **profile) as dst:
            for band_number, band_name in enumerate(OUTPUT_BANDS, start=1):
                dst.set_band_description(band_number, band_name)
            dst.update_tags(
                source_file=source_path.name,
                model_registry=str(REGISTRY_PATH),
                model_refit="false",
            )

            for window in iter_windows(src.width, src.height, BLOCK_SIZE):
                raw = src.read(required_source_indexes, window=window).astype(np.float64)
                arrays = {name: raw[i] for i, name in enumerate(required_source_features)}
                support_array = arrays[SUPPORT_BAND]
                source_nodata = src.nodata if src.nodata is not None else NODATA
                support = (
                    np.isfinite(support_array)
                    & (support_array != source_nodata)
                    & (support_array > 0)
                )

                shape = support.shape
                output = np.full((len(OUTPUT_BANDS), *shape), NODATA, dtype=np.float32)
                output[OUTPUT_BAND_INDEX[SUPPORT_BAND] - 1, support] = 1.0

                for artifact_id in PREDICTION_BANDS:
                    bundle = bundles[artifact_id]
                    valid = support.copy()
                    spectral_features = [
                        name for name in bundle["feature_names"]
                        if name in SPECTRAL_FEATURE_NAMES
                    ]
                    context_features = [
                        name for name in bundle["feature_names"]
                        if name not in SPECTRAL_FEATURE_NAMES
                    ]
                    for feature_name in spectral_features:
                        values = arrays[feature_name]
                        valid &= np.isfinite(values) & (values != source_nodata)
                    if not valid.any():
                        continue

                    frame_values = {}
                    for feature_name in bundle["feature_names"]:
                        values = arrays[feature_name][valid].astype(float)
                        values[(~np.isfinite(values)) | (values == source_nodata)] = np.nan
                        frame_values[feature_name] = values
                    feature_frame = pd.DataFrame(frame_values)
                    prediction, p_high = predict_bundle(bundle, feature_frame)
                    finite_prediction = np.isfinite(prediction)
                    prediction_positions = np.flatnonzero(valid)
                    prediction_positions = prediction_positions[finite_prediction]
                    output_flat = output[OUTPUT_BAND_INDEX[artifact_id] - 1].reshape(-1)
                    output_flat[prediction_positions] = prediction[finite_prediction].astype(np.float32)

                    if p_high is not None:
                        probability_name = f"{artifact_id}_p_high"
                        probability_flat = output[OUTPUT_BAND_INDEX[probability_name] - 1].reshape(-1)
                        probability_flat[prediction_positions] = p_high[finite_prediction].astype(np.float32)

                    ood = ood_fraction(feature_frame, bundle["feature_names"])
                    ood_flat = output[OUTPUT_BAND_INDEX[f"{artifact_id}_ood_fraction"] - 1].reshape(-1)
                    ood_flat[prediction_positions] = ood[finite_prediction].astype(np.float32)

                    if artifact_id in CONTEXT_ARTIFACT_IDS:
                        context_missing = feature_frame[context_features].isna().mean(axis=1).to_numpy(dtype=np.float32)
                        missing_flat = output[
                            OUTPUT_BAND_INDEX[f"{artifact_id}_context_missing_fraction"] - 1
                        ].reshape(-1)
                        missing_flat[prediction_positions] = context_missing[finite_prediction]

                dst.write(output, window=window)

    os.replace(partial_path, output_path)
    return output_path


def mosaic_prediction_tiles(tile_paths, final_path):
    tile_paths = [Path(path) for path in tile_paths]
    final_path = Path(final_path)
    if final_path.exists() and not OVERWRITE_OUTPUTS:
        return final_path
    partial_path = final_path.with_name(f"{final_path.stem}.partial{final_path.suffix}")
    partial_path.unlink(missing_ok=True)
    if len(tile_paths) == 1:
        shutil.copy2(tile_paths[0], partial_path)
        os.replace(partial_path, final_path)
        return final_path

    sources = [rasterio.open(path) for path in tile_paths]
    try:
        first = sources[0]
        for source in sources[1:]:
            if source.crs != first.crs or source.count != first.count:
                raise ValueError("Prediction tiles have inconsistent CRS or band counts.")
            if not np.allclose(
                [source.transform.a, source.transform.b, source.transform.d, source.transform.e],
                [first.transform.a, first.transform.b, first.transform.d, first.transform.e],
                rtol=0, atol=1e-12,
            ):
                raise ValueError("Prediction tiles have inconsistent pixel grids.")

        # Earth Engine may export large tiled rasters with a positive Y pixel size
        # (south-up orientation). rasterio.merge rejects these rasters, so place
        # every tile on its native grid using integer row/column offsets instead.
        x_size = first.transform.a
        y_size = first.transform.e
        x_origin = min(source.transform.c for source in sources) if x_size > 0 else max(source.transform.c for source in sources)
        y_origin = min(source.transform.f for source in sources) if y_size > 0 else max(source.transform.f for source in sources)
        placements = []
        for source in sources:
            col_offset = int(round((source.transform.c - x_origin) / x_size))
            row_offset = int(round((source.transform.f - y_origin) / y_size))
            placements.append((source, row_offset, col_offset))

        output_height = max(row_offset + source.height for source, row_offset, _ in placements)
        output_width = max(col_offset + source.width for source, _, col_offset in placements)
        transform = Affine(x_size, first.transform.b, x_origin, first.transform.d, y_size, y_origin)
        profile = sources[0].profile.copy()
        profile.update(
            height=output_height, width=output_width, transform=transform,
            count=len(OUTPUT_BANDS), dtype="float32", nodata=NODATA,
            compress="DEFLATE", predictor=2, tiled=True,
            blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER",
        )
        with rasterio.open(partial_path, "w", **profile) as dst:
            for band_number, band_name in enumerate(OUTPUT_BANDS, start=1):
                dst.set_band_description(band_number, band_name)
            dst.update_tags(
                source_tiles=";".join(path.name for path in tile_paths),
                model_registry=str(REGISTRY_PATH),
                model_refit="false",
            )
            for source, row_offset, col_offset in placements:
                for _, source_window in source.block_windows(1):
                    data = source.read(window=source_window)
                    destination_window = Window(
                        col_off=col_offset + source_window.col_off,
                        row_off=row_offset + source_window.row_off,
                        width=source_window.width,
                        height=source_window.height,
                    )
                    dst.write(data, window=destination_window)
    finally:
        for source in sources:
            source.close()
    os.replace(partial_path, final_path)
    return final_path

In [ ]:
# Run all logical map jobs.
run_rows = []
final_outputs = {}

for manifest_row in manifest.sort_values("map_id").itertuples(index=False):
    map_id = manifest_row.map_id
    sat_date = pd.Timestamp(manifest_row.sat_date).strftime("%Y%m%d")
    source_paths = raster_inventory.loc[raster_inventory["map_id"].eq(map_id), "file"].map(Path).tolist()
    print(f"{map_id}: {manifest_row.coordinate_waterbody} | {sat_date} | {len(source_paths)} tile(s)")

    final_path = GEOTIFF_DIR / f"{map_id}_{sat_date}_water_quality_predictions.tif"
    predicted_tiles = []
    if not final_path.exists() or OVERWRITE_OUTPUTS:
        for source_path in source_paths:
            temporary_path = TEMP_DIR / f"{source_path.stem}_predictions.tif"
            predicted_tiles.append(process_source_tile(source_path, temporary_path))
        mosaic_prediction_tiles(predicted_tiles, final_path)
    final_outputs[map_id] = final_path

    with rasterio.open(final_path) as src:
        final_valid = (
            src.count == len(OUTPUT_BANDS)
            and src.crs is not None
            and list(src.descriptions) == OUTPUT_BANDS
        )
        run_rows.append({
            "map_id": map_id,
            "coordinate_waterbody": manifest_row.coordinate_waterbody,
            "sat_date": pd.Timestamp(manifest_row.sat_date).date().isoformat(),
            "source_tile_count": len(source_paths),
            "output_path": str(final_path),
            "band_count": src.count,
            "width": src.width,
            "height": src.height,
            "crs": str(src.crs),
            "valid": final_valid,
        })

    if not KEEP_TEMPORARY_TILES:
        for temporary_path in predicted_tiles:
            Path(temporary_path).unlink(missing_ok=True)

prediction_run_manifest = pd.DataFrame(run_rows)
display(prediction_run_manifest)
prediction_run_manifest.to_csv(TABLE_DIR / "prediction_run_manifest.csv", index=False, encoding="utf-8-sig")
if not prediction_run_manifest["valid"].all():
    raise RuntimeError("At least one final prediction raster failed validation.")

## 5. Prediction and applicability summaries

These tables describe the map products. They are not new accuracy estimates because pixel-level field truth is unavailable for the complete mapped surfaces.

In [ ]:
def valid_band_values(src, band_number):
    chunks = []
    for window in iter_windows(src.width, src.height, BLOCK_SIZE):
        values = src.read(band_number, window=window)
        valid = np.isfinite(values) & (values != NODATA)
        if valid.any():
            chunks.append(values[valid].astype(np.float64))
    return np.concatenate(chunks) if chunks else np.array([], dtype=float)


summary_rows = []
applicability_rows = []
for row in prediction_run_manifest.itertuples(index=False):
    with rasterio.open(row.output_path) as src:
        support_values = valid_band_values(src, OUTPUT_BAND_INDEX[SUPPORT_BAND])
        support_count = int((support_values > 0).sum())
        support_area_km2 = support_count * EXPORT_PIXEL_SIZE_M**2 / 1_000_000

        for artifact_id in PREDICTION_BANDS:
            values = valid_band_values(src, OUTPUT_BAND_INDEX[artifact_id])
            if values.size == 0:
                summary = dict(min=np.nan, p05=np.nan, median=np.nan, mean=np.nan, p95=np.nan, max=np.nan)
            else:
                summary = {
                    "min": values.min(),
                    "p05": np.quantile(values, 0.05),
                    "median": np.median(values),
                    "mean": values.mean(),
                    "p95": np.quantile(values, 0.95),
                    "max": values.max(),
                }
            summary_rows.append({
                "map_id": row.map_id,
                "coordinate_waterbody": row.coordinate_waterbody,
                "sat_date": row.sat_date,
                "artifact_id": artifact_id,
                "target": bundles[artifact_id]["target"],
                "feature_set": bundles[artifact_id]["feature_set"],
                "valid_prediction_pixels": int(values.size),
                "support_area_km2_approx": support_area_km2,
                **summary,
            })

            ood_name = f"{artifact_id}_ood_fraction"
            ood = valid_band_values(src, OUTPUT_BAND_INDEX[ood_name])
            if artifact_id in CONTEXT_ARTIFACT_IDS:
                missing_name = f"{artifact_id}_context_missing_fraction"
                context_missing = valid_band_values(src, OUTPUT_BAND_INDEX[missing_name])
            else:
                context_missing = np.zeros_like(ood)
            applicability_rows.append({
                "map_id": row.map_id,
                "coordinate_waterbody": row.coordinate_waterbody,
                "artifact_id": artifact_id,
                "n_pixels": int(ood.size),
                "mean_ood_fraction": ood.mean() if ood.size else np.nan,
                "caution_fraction_gt_0_10": (ood > 0.10).mean() if ood.size else np.nan,
                "out_of_domain_fraction_gt_0_25": (ood > 0.25).mean() if ood.size else np.nan,
                "mean_context_missing_fraction": context_missing.mean() if context_missing.size else np.nan,
                "any_context_missing_pixel_fraction": (context_missing > 0).mean() if context_missing.size else np.nan,
            })

prediction_summary = pd.DataFrame(summary_rows)
applicability_summary = pd.DataFrame(applicability_rows)
prediction_summary.to_csv(TABLE_DIR / "prediction_summary_by_waterbody.csv", index=False, encoding="utf-8-sig")
applicability_summary.to_csv(TABLE_DIR / "applicability_domain_summary.csv", index=False, encoding="utf-8-sig")

print("Primary S2 + Context summaries")
display(
    prediction_summary.loc[prediction_summary["artifact_id"].isin([
        "turbidity_s2_context", "ss_s2_context", "do_s2_context"
    ])].round(4)
)
print("Applicability-domain summary")
display(applicability_summary.round(4))

In [ ]:
# S2-only versus S2+Context comparison at the water-body level.
comparison_rows = []
for target, s2_id, context_id in [
    ("Turbidity", "turbidity_s2", "turbidity_s2_context"),
    ("SS", "ss_s2", "ss_s2_context"),
]:
    s2_table = prediction_summary.loc[prediction_summary["artifact_id"].eq(s2_id), [
        "map_id", "coordinate_waterbody", "median", "mean"
    ]].rename(columns={"median": "s2_median", "mean": "s2_mean"})
    context_table = prediction_summary.loc[prediction_summary["artifact_id"].eq(context_id), [
        "map_id", "median", "mean"
    ]].rename(columns={"median": "s2_context_median", "mean": "s2_context_mean"})
    joined = s2_table.merge(context_table, on="map_id", how="inner")
    joined.insert(2, "target", target)
    joined["median_difference"] = joined["s2_context_median"] - joined["s2_median"]
    joined["mean_difference"] = joined["s2_context_mean"] - joined["s2_mean"]
    comparison_rows.append(joined)

feature_set_comparison = pd.concat(comparison_rows, ignore_index=True)
display(feature_set_comparison.round(4))
feature_set_comparison.to_csv(TABLE_DIR / "s2_vs_context_map_summary.csv", index=False, encoding="utf-8-sig")

In [ ]:
# S2-only versus S2+Context comparison at the water-body level.
comparison_rows = []
for target, s2_id, context_id in [
    ("Turbidity", "turbidity_s2", "turbidity_s2_context"),
    ("SS", "ss_s2", "ss_s2_context"),
]:
    s2_table = prediction_summary.loc[prediction_summary["artifact_id"].eq(s2_id), [
        "map_id", "coordinate_waterbody", "median", "mean"
    ]].rename(columns={"median": "s2_median", "mean": "s2_mean"})
    context_table = prediction_summary.loc[prediction_summary["artifact_id"].eq(context_id), [
        "map_id", "median", "mean"
    ]].rename(columns={"median": "s2_context_median", "mean": "s2_context_mean"})
    joined = s2_table.merge(context_table, on="map_id", how="inner")
    joined.insert(2, "target", target)
    joined["median_difference"] = joined["s2_context_median"] - joined["s2_median"]
    joined["mean_difference"] = joined["s2_context_mean"] - joined["s2_mean"]
    comparison_rows.append(joined)

feature_set_comparison = pd.concat(comparison_rows, ignore_index=True)
display(feature_set_comparison.round(4))
feature_set_comparison.to_csv(TABLE_DIR / "s2_vs_context_map_summary.csv", index=False, encoding="utf-8-sig")

## 6. Prediction Maps on Exact-Date Sentinel-2 True Color

Each prediction is displayed semi-transparently over the matching Sentinel-2 RGB exported by the polygon Notebook 03. Identical target-specific color scales are retained across water bodies. The saved prediction values are not altered by visualization.

In [ ]:
NOTO_FONT_DIR = Path("/usr/share/fonts/truetype/noto")
font_paths = sorted(NOTO_FONT_DIR.glob("NotoSans*.ttf"))
if not font_paths:
    raise FileNotFoundError(
        "Noto fonts are missing. Re-run the Google Colab setup cell before plotting."
    )
for font_path in font_paths:
    font_manager.fontManager.addfont(str(font_path))
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Noto Sans", "Noto Sans Thai", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def read_preview(src, band_number, max_dimension=1200):
    scale = max(src.width / max_dimension, src.height / max_dimension, 1)
    out_width = max(1, int(src.width / scale))
    out_height = max(1, int(src.height / scale))
    values = src.read(
        band_number,
        out_shape=(out_height, out_width),
        resampling=rasterio.enums.Resampling.nearest,
    ).astype(float)
    values[(~np.isfinite(values)) | (values == NODATA)] = np.nan
    preview_transform = src.transform * Affine.scale(
        src.width / out_width, src.height / out_height
    )
    return values, preview_transform


def preview_extent(transform, width, height):
    x0, y0 = transform * (0, 0)
    x1, y1 = transform * (width, height)
    return (min(x0, x1), max(x0, x1), min(y0, y1), max(y0, y1))


def build_sentinel2_rgb_preview(map_id, prediction_src, out_height, out_width, destination_transform):
    paths = rgb_inventory.loc[
        rgb_inventory["map_id"].eq(map_id), "file"
    ].map(Path).tolist()
    rgb = np.full((3, out_height, out_width), np.nan, dtype=np.float32)
    for path in paths:
        with rasterio.open(path) as source:
            for band_index in range(1, 4):
                temporary = np.full((out_height, out_width), np.nan, dtype=np.float32)
                reproject(
                    source=rasterio.band(source, band_index), destination=temporary,
                    src_transform=source.transform, src_crs=source.crs,
                    src_nodata=source.nodata, dst_transform=destination_transform,
                    dst_crs=prediction_src.crs, dst_nodata=np.nan,
                    resampling=Resampling.bilinear,
                )
                fill = np.isfinite(temporary) & ~np.isfinite(rgb[band_index - 1])
                rgb[band_index - 1, fill] = temporary[fill]

    rgb = np.moveaxis(rgb, 0, -1)
    valid = np.isfinite(rgb).all(axis=2)
    output = np.zeros_like(rgb, dtype=float)
    if valid.any():
        for channel in range(3):
            values = rgb[:, :, channel][valid]
            low, high = np.quantile(values, [0.02, 0.98])
            if high <= low:
                high = low + 1e-6
            output[:, :, channel] = np.clip(
                (rgb[:, :, channel] - low) / (high - low), 0, 1
            )
        output = np.power(output, 0.85)
    return output, valid


def add_satellite_background(axis, crs, bounds):
    left, bottom, right, top = bounds
    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)
    try:
        ctx.add_basemap(
            axis, crs=crs, source=BASEMAP_SOURCE,
            zoom=BASEMAP_ZOOM, attribution=False, reset_extent=False,
        )
    except Exception as exc:
        axis.set_facecolor("#d9d9d9")
        print("Basemap warning:", exc)


def prediction_support_bounds(support, transform, geometry, map_id):
    """Return bounds of valid prediction pixels inside one reviewed polygon part."""
    support_mask = np.isfinite(support) & (support > 0)
    polygon_mask = geometry_mask(
        [mapping(geometry)],
        out_shape=support.shape,
        transform=transform,
        invert=True,
        all_touched=True,
    )
    selected = support_mask & polygon_mask
    rows, cols = np.where(selected)
    support_pixels = int(selected.sum())
    if support_pixels < MIN_QUICKLOOK_SUPPORT_PIXELS:
        return None, support_pixels

    row_min, row_max = int(rows.min()), int(rows.max()) + 1
    col_min, col_max = int(cols.min()), int(cols.max()) + 1
    corners = [
        transform * (col_min, row_min),
        transform * (col_max, row_min),
        transform * (col_min, row_max),
        transform * (col_max, row_max),
    ]
    xs = [point[0] for point in corners]
    ys = [point[1] for point in corners]
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    pixel_size = max(abs(transform.a), abs(transform.e))
    padding_fraction = PREDICTION_VIEW_PADDING_BY_MAP_ID.get(
        str(map_id), PREDICTION_VIEW_PADDING_FRACTION
    )
    padding = max(
        (right - left) * padding_fraction,
        (top - bottom) * padding_fraction,
        pixel_size * PREDICTION_VIEW_MIN_PADDING_PIXELS,
    )
    return (left - padding, bottom - padding, right + padding, top + padding), support_pixels


def add_support_outline(axis, support, extent, origin):
    support_mask = np.isfinite(support) & (support > 0)
    if support_mask.any() and (~support_mask).any():
        axis.contour(
            support_mask.astype(float), levels=[0.5], colors=["#00FFFF"],
            linewidths=0.45, extent=extent, origin=origin, zorder=4,
        )


def draw_quicklook(row, src, arrays, rgb, rgb_valid, transform, bounds, suffix, part_label):
    extent = preview_extent(transform, arrays["tur_ctx"].shape[1], arrays["tur_ctx"].shape[0])
    origin = "upper" if transform.e < 0 else "lower"
    rgba = np.dstack([rgb, rgb_valid.astype(float) * 0.92])

    fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
    panels = [
        (arrays["tur_ctx"], "Turbidity: S2 + Context", "viridis", target_norm("turbidity_s2_context")),
        (arrays["ss_ctx"], "SS: S2 + Context", "plasma", target_norm("ss_s2_context")),
        (arrays["do_ctx"], "DO proxy: S2 + Context", "cividis", target_norm("do_s2_context")),
    ]
    for axis, (values, title, cmap, norm) in zip(axes[0], panels):
        add_satellite_background(axis, src.crs, bounds)
        axis.imshow(rgba, extent=extent, origin=origin, interpolation="bilinear", zorder=2)
        image = axis.imshow(
            values, cmap=cmap, norm=norm, alpha=0.68,
            extent=extent, origin=origin, zorder=3,
        )
        add_support_outline(axis, arrays["support"], extent, origin)
        axis.set_title(title, fontsize=11)
        axis.axis("off")
        fig.colorbar(image, ax=axis, shrink=0.78)

    differences = [
        (arrays["tur_ctx"] - arrays["tur_s2"], "Turbidity: Context - S2"),
        (arrays["ss_ctx"] - arrays["ss_s2"], "SS: Context - S2"),
    ]
    for axis, (values, title) in zip(axes[1, :2], differences):
        finite = np.abs(values[np.isfinite(values)])
        limit = np.quantile(finite, 0.99) if finite.size else 1.0
        limit = max(float(limit), 1e-6)
        add_satellite_background(axis, src.crs, bounds)
        axis.imshow(rgba, extent=extent, origin=origin, interpolation="bilinear", zorder=2)
        image = axis.imshow(
            values, cmap="RdBu_r",
            norm=TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit),
            alpha=0.68, extent=extent, origin=origin, zorder=3,
        )
        add_support_outline(axis, arrays["support"], extent, origin)
        axis.set_title(title, fontsize=11)
        axis.axis("off")
        fig.colorbar(image, ax=axis, shrink=0.78)

    axis = axes[1, 2]
    add_satellite_background(axis, src.crs, bounds)
    axis.imshow(rgba, extent=extent, origin=origin, interpolation="bilinear", zorder=2)
    image = axis.imshow(
        arrays["max_ood"], cmap="magma", vmin=0, vmax=1, alpha=0.68,
        extent=extent, origin=origin, zorder=3,
    )
    add_support_outline(axis, arrays["support"], extent, origin)
    axis.set_title("Maximum OOD-feature fraction", fontsize=11)
    axis.axis("off")
    fig.colorbar(image, ax=axis, shrink=0.78)

    for axis in axes.flat:
        axis.set_xlim(bounds[0], bounds[2])
        axis.set_ylim(bounds[1], bounds[3])
        axis.set_aspect("equal", adjustable="box")

    title = f"{row.map_id} | {row.coordinate_waterbody} | {row.sat_date} | {part_label}"
    fig.suptitle(title, fontsize=14)
    output = QUICKLOOK_DIR / f"{row.map_id}_{row.sat_date.replace('-', '')}_{suffix}_quicklook.png"
    fig.savefig(output, dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    return output


# Global target display ranges preserve comparability across all maps and polygon parts.
display_samples = {artifact_id: [] for artifact_id in PREDICTION_BANDS}
for path in prediction_run_manifest["output_path"]:
    with rasterio.open(path) as src:
        for artifact_id in PREDICTION_BANDS:
            values, _ = read_preview(src, OUTPUT_BAND_INDEX[artifact_id], max_dimension=600)
            finite = values[np.isfinite(values)]
            if finite.size:
                display_samples[artifact_id].append(finite)

display_limits = {}
for artifact_id, chunks in display_samples.items():
    values = np.concatenate(chunks) if chunks else np.array([0.0, 1.0])
    display_limits[artifact_id] = (
        float(np.quantile(values, 0.01)), float(np.quantile(values, 0.99))
    )


def target_norm(artifact_id):
    vmin, vmax = display_limits[artifact_id]
    if bundles[artifact_id]["target"] in {"Turbidity", "SS"}:
        positive = [value for value in [vmin, vmax] if value > 0]
        lower = max(min(positive) if positive else 0.01, 0.01)
        return LogNorm(vmin=lower, vmax=max(vmax, lower * 1.01))
    return Normalize(vmin=vmin, vmax=max(vmax, vmin + 1e-6))


quicklook_rows = []
for row in prediction_run_manifest.itertuples(index=False):
    with rasterio.open(row.output_path) as src:
        arrays = {}
        arrays["tur_s2"], transform = read_preview(src, OUTPUT_BAND_INDEX["turbidity_s2"])
        arrays["tur_ctx"], _ = read_preview(src, OUTPUT_BAND_INDEX["turbidity_s2_context"])
        arrays["ss_s2"], _ = read_preview(src, OUTPUT_BAND_INDEX["ss_s2"])
        arrays["ss_ctx"], _ = read_preview(src, OUTPUT_BAND_INDEX["ss_s2_context"])
        arrays["do_ctx"], _ = read_preview(src, OUTPUT_BAND_INDEX["do_s2_context"])
        arrays["support"], _ = read_preview(src, OUTPUT_BAND_INDEX[SUPPORT_BAND])
        ood_arrays = [
            read_preview(src, OUTPUT_BAND_INDEX[f"{artifact_id}_ood_fraction"])[0]
            for artifact_id in ["turbidity_s2_context", "ss_s2_context", "do_s2_context"]
        ]
        arrays["max_ood"] = np.nanmax(np.stack(ood_arrays), axis=0)
        rgb, rgb_valid = build_sentinel2_rgb_preview(
            row.map_id, src, arrays["tur_ctx"].shape[0], arrays["tur_ctx"].shape[1], transform
        )

        geometry_row = waterbody_polygons.loc[waterbody_polygons["map_id"].eq(row.map_id)]
        geometry = geometry_row.to_crs(src.crs).geometry.iloc[0]
        parts = list(geometry.geoms) if geometry.geom_type == "MultiPolygon" else [geometry]

        if len(parts) == 1 or DISPLAY_OVERVIEW_FOR_MULTIPOLYGON:
            overview_bounds, support_pixels = prediction_support_bounds(
                arrays["support"], transform, geometry, row.map_id
            )
            if overview_bounds is not None:
                output = draw_quicklook(
                    row, src, arrays, rgb, rgb_valid, transform,
                    overview_bounds, "overview", "overview",
                )
                quicklook_rows.append({
                    "map_id": row.map_id, "view": "overview", "part": None,
                    "support_pixels": support_pixels, "status": "created",
                    "output_path": str(output),
                })
            else:
                quicklook_rows.append({
                    "map_id": row.map_id, "view": "overview", "part": None,
                    "support_pixels": support_pixels, "status": "skipped_no_prediction_support",
                    "output_path": None,
                })

        if len(parts) > 1:
            for part_index, part in enumerate(parts, start=1):
                bounds, support_pixels = prediction_support_bounds(
                    arrays["support"], transform, part, row.map_id
                )
                if bounds is None:
                    quicklook_rows.append({
                        "map_id": row.map_id, "view": "polygon_part", "part": part_index,
                        "support_pixels": support_pixels, "status": "skipped_no_prediction_support",
                        "output_path": None,
                    })
                    continue
                output = draw_quicklook(
                    row, src, arrays, rgb, rgb_valid, transform,
                    bounds, f"part{part_index:02d}", f"polygon {part_index}/{len(parts)}",
                )
                quicklook_rows.append({
                    "map_id": row.map_id, "view": "polygon_part", "part": part_index,
                    "support_pixels": support_pixels, "status": "created",
                    "output_path": str(output),
                })

quicklook_manifest = pd.DataFrame(quicklook_rows)
quicklook_manifest.to_csv(TABLE_DIR / "quicklook_manifest.csv", index=False, encoding="utf-8-sig")
display(quicklook_manifest)


## 7. Final completion gate

Passing this gate means the locked artifacts were applied to all selected water bodies, every final raster has the expected output schema, and each artifact produced supported predictions. It does not constitute independent validation of map accuracy.

In [ ]:
gate_rows = []
gate_rows.append({
    "check": "notebook03_critical_checks_ready",
    "passed": bool(NOTEBOOK03_CRITICAL_READY),
    "detail": f"{int(blocking_rows['passed_bool'].sum())}/{len(blocking_rows)} critical checks",
})
gate_rows.append({
    "check": "sentinel2_rgb_available_for_every_map",
    "passed": set(manifest["map_id"]).issubset(set(rgb_inventory["map_id"])),
    "detail": f"{rgb_inventory['map_id'].nunique()}/{len(manifest)} logical maps",
})
gate_rows.append({
    "check": "five_locked_artifacts_loaded",
    "passed": len(bundles) == 5,
    "detail": ", ".join(sorted(bundles)),
})
gate_rows.append({
    "check": "all_manifest_waterbodies_predicted",
    "passed": set(prediction_run_manifest["map_id"]) == set(manifest["map_id"]),
    "detail": f"{len(prediction_run_manifest)}/{len(manifest)} logical maps",
})
gate_rows.append({
    "check": "all_final_geotiffs_valid",
    "passed": bool(prediction_run_manifest["valid"].all()),
    "detail": f"{prediction_run_manifest['valid'].sum()}/{len(prediction_run_manifest)} rasters",
})
expected_summary_rows = len(manifest) * len(PREDICTION_BANDS)
gate_rows.append({
    "check": "all_prediction_summaries_available",
    "passed": len(prediction_summary) == expected_summary_rows and prediction_summary["valid_prediction_pixels"].gt(0).all(),
    "detail": f"{len(prediction_summary)}/{expected_summary_rows} rows",
})
gate_rows.append({
    "check": "no_model_refit_in_notebook04",
    "passed": True,
    "detail": "Artifacts loaded with joblib; no estimator.fit() call",
})

notebook04_readiness = pd.DataFrame(gate_rows)
display(notebook04_readiness)
notebook04_readiness.to_csv(TABLE_DIR / "notebook04_readiness.csv", index=False, encoding="utf-8-sig")

NOTEBOOK_04_COMPLETE = bool(notebook04_readiness["passed"].all())
print("NOTEBOOK_04_COMPLETE =", NOTEBOOK_04_COMPLETE)
print("Prediction rasters:", GEOTIFF_DIR)
print("Quicklooks       :", QUICKLOOK_DIR)
print("Summary tables   :", TABLE_DIR)
assert NOTEBOOK_04_COMPLETE

In [ ]:
# WB02 English-only static quicklook.
from types import SimpleNamespace

WB02_MAP_ID = "WB02"
WB02_ENGLISH_NAME = "Thale Luang"


def build_rgb_preview_from_paths(paths, prediction_src, out_height, out_width, destination_transform):
    rgb = np.full((3, out_height, out_width), np.nan, dtype=np.float32)
    for path in paths:
        with rasterio.open(path) as source:
            for band_index in range(1, 4):
                temporary = np.full((out_height, out_width), np.nan, dtype=np.float32)
                reproject(
                    source=rasterio.band(source, band_index), destination=temporary,
                    src_transform=source.transform, src_crs=source.crs,
                    src_nodata=source.nodata, dst_transform=destination_transform,
                    dst_crs=prediction_src.crs, dst_nodata=np.nan,
                    resampling=Resampling.bilinear,
                )
                fill = np.isfinite(temporary) & ~np.isfinite(rgb[band_index - 1])
                rgb[band_index - 1, fill] = temporary[fill]

    rgb = np.moveaxis(rgb, 0, -1)
    valid = np.isfinite(rgb).all(axis=2)
    output = np.zeros_like(rgb, dtype=float)
    if valid.any():
        for channel in range(3):
            values = rgb[:, :, channel][valid]
            low, high = np.quantile(values, [0.02, 0.98])
            if high <= low:
                high = low + 1e-6
            output[:, :, channel] = np.clip(
                (rgb[:, :, channel] - low) / (high - low), 0, 1
            )
        output = np.power(output, 0.85)
    return output, valid


# Read the unfiltered Notebook 03 manifest so this cell still works when the main
# MAP_IDS_TO_RUN configuration excludes WB02.
wb02_manifest = pd.read_csv(MANIFEST_PATH)
wb02_manifest["map_id"] = wb02_manifest["map_id"].astype(str)
wb02_match = wb02_manifest.loc[wb02_manifest["map_id"].eq(WB02_MAP_ID)]
if len(wb02_match) != 1:
    raise ValueError(f"Expected exactly one manifest row for {WB02_MAP_ID}; found {len(wb02_match)}")

wb02_date = pd.Timestamp(wb02_match.iloc[0]["sat_date"]).date().isoformat()
wb02_date_token = wb02_date.replace("-", "")
WB02_PREDICTION_PATH = GEOTIFF_DIR / f"{WB02_MAP_ID}_{wb02_date_token}_water_quality_predictions.tif"
if not WB02_PREDICTION_PATH.exists():
    raise FileNotFoundError(
        f"Missing {WB02_PREDICTION_PATH}. Run the prediction section with "
        f"MAP_IDS_TO_RUN = ['{WB02_MAP_ID}'] first."
    )

wb02_rgb_paths = []
for path in sorted(RGB_INPUT_ROOT.rglob("*.tif")):
    identity = parse_export_identity(path, "sentinel2_rgb")
    if identity == (WB02_MAP_ID, wb02_date_token):
        wb02_rgb_paths.append(path)
if not wb02_rgb_paths:
    raise FileNotFoundError(
        f"No exact-date Sentinel-2 RGB GeoTIFF was found for {WB02_MAP_ID} on {wb02_date}."
    )

with rasterio.open(WB02_PREDICTION_PATH) as wb02_src:
    wb02_arrays = {}
    wb02_arrays["tur_s2"], wb02_transform = read_preview(
        wb02_src, OUTPUT_BAND_INDEX["turbidity_s2"]
    )
    wb02_arrays["tur_ctx"], _ = read_preview(
        wb02_src, OUTPUT_BAND_INDEX["turbidity_s2_context"]
    )
    wb02_arrays["ss_s2"], _ = read_preview(wb02_src, OUTPUT_BAND_INDEX["ss_s2"])
    wb02_arrays["ss_ctx"], _ = read_preview(
        wb02_src, OUTPUT_BAND_INDEX["ss_s2_context"]
    )
    wb02_arrays["do_ctx"], _ = read_preview(
        wb02_src, OUTPUT_BAND_INDEX["do_s2_context"]
    )
    wb02_arrays["support"], _ = read_preview(wb02_src, OUTPUT_BAND_INDEX[SUPPORT_BAND])
    wb02_ood_arrays = [
        read_preview(wb02_src, OUTPUT_BAND_INDEX[f"{artifact_id}_ood_fraction"])[0]
        for artifact_id in ["turbidity_s2_context", "ss_s2_context", "do_s2_context"]
    ]
    wb02_arrays["max_ood"] = np.nanmax(np.stack(wb02_ood_arrays), axis=0)
    wb02_rgb, wb02_rgb_valid = build_rgb_preview_from_paths(
        wb02_rgb_paths, wb02_src,
        wb02_arrays["tur_ctx"].shape[0], wb02_arrays["tur_ctx"].shape[1],
        wb02_transform,
    )
    wb02_crs = wb02_src.crs

    wb02_geometry = (
        waterbody_polygons.loc[waterbody_polygons["map_id"].eq(WB02_MAP_ID)]
        .to_crs(wb02_src.crs).geometry.iloc[0]
    )
    wb02_parts = (
        list(wb02_geometry.geoms)
        if wb02_geometry.geom_type == "MultiPolygon"
        else [wb02_geometry]
    )

    wb02_part_rows = []
    for part_index, part in enumerate(wb02_parts, start=1):
        bounds, support_pixels = prediction_support_bounds(
            wb02_arrays["support"], wb02_transform, part, WB02_MAP_ID
        )
        wb02_part_rows.append({
            "part_index": part_index,
            "support_pixels": support_pixels,
            "bounds": bounds,
            "geometry": part,
        })

    valid_parts = [row for row in wb02_part_rows if row["bounds"] is not None]
    if not valid_parts:
        raise RuntimeError("WB02 has no polygon part containing prediction-support pixels.")
    wb02_selected = max(valid_parts, key=lambda row: row["support_pixels"])
    wb02_selected_part = wb02_selected["geometry"]
    wb02_bounds = wb02_selected["bounds"]
    wb02_selected_part_index = wb02_selected["part_index"]

    wb02_row_english = SimpleNamespace(
        map_id=WB02_MAP_ID,
        coordinate_waterbody=WB02_ENGLISH_NAME,
        sat_date=wb02_date,
    )
    WB02_STATIC_OUTPUT = draw_quicklook(
        wb02_row_english, wb02_src, wb02_arrays,
        wb02_rgb, wb02_rgb_valid, wb02_transform, wb02_bounds,
        "english_best_part",
        f"polygon {wb02_selected_part_index}/{len(wb02_parts)}",
    )

display(pd.DataFrame([
    {
        "map_id": WB02_MAP_ID,
        "waterbody": WB02_ENGLISH_NAME,
        "satellite_date": wb02_date,
        "selected_polygon_part": wb02_selected_part_index,
        "polygon_part_count": len(wb02_parts),
        "prediction_support_pixels": wb02_selected["support_pixels"],
        "static_output": str(WB02_STATIC_OUTPUT),
    }
]))


In [ ]:
# WB02 interactive map with zoom and switchable prediction layers.
import base64
from io import BytesIO

import folium
from affine import Affine
from folium.raster_layers import ImageOverlay
from PIL import Image
from rasterio.warp import (
    Resampling,
    calculate_default_transform,
    reproject,
)


def affine_raster_bounds(transform, width, height):
    """Return ordered bounds for both north-up and south-up rasters."""
    corners = [
        transform * (0, 0),
        transform * (width, 0),
        transform * (0, height),
        transform * (width, height),
    ]

    xs = [point[0] for point in corners]
    ys = [point[1] for point in corners]

    return min(xs), min(ys), max(xs), max(ys)


def crop_preview_to_bounds(values, transform, bounds):
    """Crop using inverse affine coordinates."""
    height, width = values.shape[:2]
    left, bottom, right, top = bounds

    inverse_transform = ~transform

    pixel_corners = [
        inverse_transform * (left, bottom),
        inverse_transform * (left, top),
        inverse_transform * (right, bottom),
        inverse_transform * (right, top),
    ]

    columns = [point[0] for point in pixel_corners]
    rows = [point[1] for point in pixel_corners]

    col_start = max(0, int(np.floor(min(columns))))
    row_start = max(0, int(np.floor(min(rows))))
    col_stop = min(width, int(np.ceil(max(columns))))
    row_stop = min(height, int(np.ceil(max(rows))))

    if col_stop <= col_start or row_stop <= row_start:
        raise ValueError(
            "The WB02 display bounds do not overlap the prediction preview. "
            f"bounds={bounds}, transform={transform}"
        )

    cropped = values[
        row_start:row_stop,
        col_start:col_stop,
    ]

    cropped_transform = (
        transform * Affine.translation(col_start, row_start)
    )

    return cropped, cropped_transform


def reproject_preview_to_wgs84(
    values,
    source_transform,
    source_crs,
    resampling,
):
    height, width = values.shape

    west, south, east, north = affine_raster_bounds(
        source_transform,
        width,
        height,
    )

    destination_transform, destination_width, destination_height = (
        calculate_default_transform(
            source_crs,
            "EPSG:4326",
            width,
            height,
            west,
            south,
            east,
            north,
        )
    )

    destination = np.full(
        (destination_height, destination_width),
        np.nan,
        dtype=np.float32,
    )

    reproject(
        source=values.astype(np.float32),
        destination=destination,
        src_transform=source_transform,
        src_crs=source_crs,
        src_nodata=np.nan,
        dst_transform=destination_transform,
        dst_crs="EPSG:4326",
        dst_nodata=np.nan,
        resampling=resampling,
    )

    west, south, east, north = affine_raster_bounds(
        destination_transform,
        destination_width,
        destination_height,
    )

    return destination, [[south, west], [north, east]]

def rgba_to_data_url(rgba):
    buffer = BytesIO()
    Image.fromarray(rgba.astype(np.uint8), mode="RGBA").save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


def prediction_data_url(values, cmap_name, norm, opacity=0.72):
    valid = np.isfinite(values)
    normalized = np.zeros(values.shape, dtype=float)
    normalized[valid] = np.clip(norm(values[valid]), 0, 1)
    rgba = plt.colormaps[cmap_name](normalized, bytes=True)
    rgba[..., 3] = np.where(valid, int(round(255 * opacity)), 0).astype(np.uint8)
    return rgba_to_data_url(rgba)


# Crop every layer to the same selected WB02 prediction frame.
wb02_wgs84_layers = {}
wb02_overlay_bounds = None
for layer_name, values, resampling_method in [
    ("Turbidity", wb02_arrays["tur_ctx"], Resampling.bilinear),
    ("SS", wb02_arrays["ss_ctx"], Resampling.bilinear),
    ("DO proxy", wb02_arrays["do_ctx"], Resampling.bilinear),
    ("Maximum OOD fraction", wb02_arrays["max_ood"], Resampling.nearest),
]:
    cropped, cropped_transform = crop_preview_to_bounds(values, wb02_transform, wb02_bounds)
    projected, layer_bounds = reproject_preview_to_wgs84(
        cropped, cropped_transform, wb02_crs, resampling_method
    )
    wb02_wgs84_layers[layer_name] = projected
    wb02_overlay_bounds = layer_bounds

rgb_channels = []
for channel in range(3):
    cropped, cropped_transform = crop_preview_to_bounds(
        wb02_rgb[:, :, channel], wb02_transform, wb02_bounds
    )
    projected, rgb_bounds = reproject_preview_to_wgs84(
        cropped, cropped_transform, wb02_crs, Resampling.bilinear
    )
    rgb_channels.append(projected)
rgb_wgs84 = np.stack(rgb_channels, axis=-1)
rgb_valid_wgs84 = np.isfinite(rgb_wgs84).all(axis=2)
rgb_rgba = np.zeros((*rgb_wgs84.shape[:2], 4), dtype=np.uint8)
rgb_rgba[:, :, :3] = np.clip(np.nan_to_num(rgb_wgs84), 0, 1) * 255
rgb_rgba[:, :, 3] = np.where(rgb_valid_wgs84, 230, 0).astype(np.uint8)

center_lat = (wb02_overlay_bounds[0][0] + wb02_overlay_bounds[1][0]) / 2
center_lon = (wb02_overlay_bounds[0][1] + wb02_overlay_bounds[1][1]) / 2
wb02_interactive_map = folium.Map(
    location=[center_lat, center_lon], zoom_start=12,
    tiles=None, control_scale=True,
)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Esri World Imagery", overlay=False, control=True, show=True,
).add_to(wb02_interactive_map)

ImageOverlay(
    image=rgba_to_data_url(rgb_rgba), bounds=rgb_bounds,
    name=f"Sentinel-2 RGB ({wb02_date})", opacity=1.0,
    interactive=True, cross_origin=False, zindex=2, show=True,
).add_to(wb02_interactive_map)

interactive_layers = [
    ("Turbidity: S2 + Context (NTU)", "Turbidity", "viridis", target_norm("turbidity_s2_context"), True),
    ("SS: S2 + Context (mg/L)", "SS", "plasma", target_norm("ss_s2_context"), False),
    ("DO proxy: S2 + Context (mg/L)", "DO proxy", "cividis", target_norm("do_s2_context"), False),
    ("Maximum OOD-feature fraction", "Maximum OOD fraction", "magma", Normalize(vmin=0, vmax=1), False),
]
for display_name, key, cmap_name, norm, show in interactive_layers:
    ImageOverlay(
        image=prediction_data_url(wb02_wgs84_layers[key], cmap_name, norm),
        bounds=wb02_overlay_bounds, name=display_name,
        opacity=1.0, interactive=True, cross_origin=False,
        zindex=3, show=show,
    ).add_to(wb02_interactive_map)

selected_part_wgs84 = gpd.GeoSeries(
    [wb02_selected_part], crs=wb02_crs
).to_crs("EPSG:4326")
folium.GeoJson(
    data=selected_part_wgs84.__geo_interface__,
    name="Reviewed polygon boundary",
    style_function=lambda feature: {
        "color": "#00FFFF", "weight": 2, "fill": False, "opacity": 0.95,
    },
    show=True,
).add_to(wb02_interactive_map)

folium.LayerControl(collapsed=False).add_to(wb02_interactive_map)
wb02_interactive_map.fit_bounds(wb02_overlay_bounds, padding=(12, 12))

INTERACTIVE_DIR = OUTPUT_ROOT / "interactive"
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)
WB02_INTERACTIVE_OUTPUT = (
    INTERACTIVE_DIR / f"{WB02_MAP_ID}_{wb02_date_token}_english_interactive.html"
)
wb02_interactive_map.save(str(WB02_INTERACTIVE_OUTPUT))
print("Interactive HTML:", WB02_INTERACTIVE_OUTPUT)
wb02_interactive_map


In [ ]:
# Prepare the three representative maps for the publication figure.
# This reads existing prediction/RGB GeoTIFFs only; it does not run a model again.

from matplotlib.patches import Rectangle


REPRESENTATIVE_SITE_CONFIG = [
    {"map_id": "WB02", "name": "Thale Luang", "letter": "A"},
    {"map_id": "WB07", "name": "Lower Lang Suan River", "letter": "B"},
    {"map_id": "WB05", "name": "Pak Phanang River", "letter": "C"},
]


def add_light_satellite_background(
    axis, crs, bounds, fallback_rgb, fallback_valid, extent, origin
):
    """Draw a pale satellite background across the complete map frame."""
    left, bottom, right, top = bounds
    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)

    try:
        ctx.add_basemap(
            axis,
            crs=crs,
            source=BASEMAP_SOURCE,
            zoom=BASEMAP_ZOOM,
            attribution=False,
            reset_extent=False,
            zorder=0,
        )
    except Exception as exc:
        axis.set_facecolor("#D8DDD8")
        print(f"Basemap warning: {exc}")

    rgb = np.asarray(fallback_rgb, dtype=float)
    valid = np.asarray(fallback_valid, dtype=bool)
    if rgb.ndim == 3 and rgb.shape[2] == 3 and valid.shape == rgb.shape[:2]:
        rgba = np.dstack([rgb, valid.astype(float) * 0.60])
        axis.imshow(
            rgba,
            extent=extent,
            origin=origin,
            interpolation="bilinear",
            zorder=1,
        )

    # A white veil leaves the satellite context visible without competing
    # with the prediction layer.
    axis.add_patch(
        Rectangle(
            (left, bottom),
            right - left,
            top - bottom,
            facecolor="white",
            edgecolor="none",
            alpha=0.40,
            zorder=2,
        )
    )
    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)


def representative_manifest_row(map_id):
    matches = prediction_run_manifest.loc[
        prediction_run_manifest["map_id"].astype(str).eq(str(map_id))
    ].copy()
    if matches.empty:
        raise ValueError(f"No completed prediction was found for {map_id}.")
    if len(matches) > 1:
        matches = matches.sort_values("sat_date")
        print(f"{map_id}: multiple predictions found; using {matches.iloc[-1]['sat_date']}")
    return matches.iloc[-1]


representative_data = []

for config in REPRESENTATIVE_SITE_CONFIG:
    row = representative_manifest_row(config["map_id"])
    prediction_path = Path(row["output_path"])
    if not prediction_path.exists():
        raise FileNotFoundError(prediction_path)

    with rasterio.open(prediction_path) as src:
        turbidity, transform = read_preview(
            src, OUTPUT_BAND_INDEX["turbidity_s2_context"]
        )
        ss, _ = read_preview(src, OUTPUT_BAND_INDEX["ss_s2_context"])
        do, _ = read_preview(src, OUTPUT_BAND_INDEX["do_s2_context"])
        support_values, _ = read_preview(src, OUTPUT_BAND_INDEX[SUPPORT_BAND])

        rgb, rgb_valid = build_sentinel2_rgb_preview(
            config["map_id"],
            src,
            turbidity.shape[0],
            turbidity.shape[1],
            transform,
        )

        geometry_rows = waterbody_polygons.loc[
            waterbody_polygons["map_id"].astype(str).eq(config["map_id"])
        ]
        if geometry_rows.empty:
            raise ValueError(f"Polygon geometry is missing for {config['map_id']}.")
        geometry = geometry_rows.to_crs(src.crs).geometry.iloc[0]

        polygon_mask = geometry_mask(
            [mapping(geometry)],
            out_shape=support_values.shape,
            transform=transform,
            invert=True,
            all_touched=True,
        )
        display_support = (
            np.isfinite(support_values)
            & (support_values > 0)
            & polygon_mask
        )

        bounds, support_pixels = prediction_support_bounds(
            support_values,
            transform,
            geometry,
            config["map_id"],
        )
        if bounds is None:
            raise ValueError(
                f"No prediction-support pixels were found for {config['map_id']}."
            )

        extent = preview_extent(
            transform,
            turbidity.shape[1],
            turbidity.shape[0],
        )
        origin = "upper" if transform.e < 0 else "lower"

    representative_data.append(
        {
            "map_id": config["map_id"],
            "name": config["name"],
            "letter": config["letter"],
            "date": pd.Timestamp(row["sat_date"]).strftime("%Y-%m-%d"),
            "prediction_path": prediction_path,
            "bounds": tuple(bounds),
            "extent": extent,
            "origin": origin,
            "rgb": rgb,
            "rgb_valid": rgb_valid,
            "display_support": display_support,
            "support_pixels": int(support_pixels),
            "arrays": {
                "turbidity": turbidity,
                "ss": ss,
                "do": do,
            },
        }
    )


representative_audit = pd.DataFrame(
    [
        {
            "letter": item["letter"],
            "map_id": item["map_id"],
            "name": item["name"],
            "date": item["date"],
            "support_pixels": item["support_pixels"],
            "array_shape": item["arrays"]["turbidity"].shape,
        }
        for item in representative_data
    ]
)

display(representative_audit)
print("representative_data ready:", len(representative_data), "maps")


In [ ]:
import math
from pathlib import Path
from urllib.request import urlretrieve

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MaxNLocator
from pyproj import Geod, Transformer
from scipy.ndimage import label as ndi_label


# This cell expects the notebook's existing objects/functions to be available:
# OUTPUT_ROOT, PROJECT_DIR, GEOTIFF_DIR, representative_data, and
# add_light_satellite_background.

FINAL_FIGURE_DIR = OUTPUT_ROOT / "publication_figures"
FINAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_CMAP = LinearSegmentedColormap.from_list(
    "unified_water_quality",
    ["#30123B", "#4662D7", "#35B779", "#FDE725"],
)
UNIFIED_CMAP.set_bad(alpha=0)


def target_values(key, nonnegative=False):
    chunks = []
    for item in representative_data:
        values = np.asarray(item["arrays"][key], dtype=float)
        values = values[np.isfinite(values)]
        if nonnegative:
            values = values[values >= 0]
        if values.size:
            chunks.append(values)
    if not chunks:
        raise ValueError(f"No valid values for {key}")
    return np.concatenate(chunks)


def _nice_step(raw_step):
    if raw_step <= 0:
        return 1.0
    magnitude = 10 ** math.floor(math.log10(raw_step))
    residual = raw_step / magnitude
    if residual <= 1:
        nice = 1
    elif residual <= 2:
        nice = 2
    elif residual <= 2.5:
        nice = 2.5
    elif residual <= 5:
        nice = 5
    else:
        nice = 10
    return nice * magnitude


def nice_scale(values, target_ticks=5, force_zero_min=True):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    vmax_data = float(values.max())
    vmin_data = float(values.min())
    vmin = 0.0 if force_zero_min else vmin_data
    step = _nice_step(max(vmax_data - vmin, 1e-9) / target_ticks)
    vmax = math.ceil(vmax_data / step) * step
    vmin = math.floor(vmin / step) * step
    ticks = np.arange(vmin, vmax + step / 2, step)
    return vmin, vmax, ticks, step


def format_ticks(ticks, step):
    if float(step).is_integer():
        return [f"{value:.0f}" for value in ticks]
    return [f"{value:.1f}" for value in ticks]


tur_vmin, tur_vmax, tur_ticks, tur_step = nice_scale(
    target_values("turbidity", nonnegative=True), force_zero_min=True
)
ss_vmin, ss_vmax, ss_ticks, ss_step = nice_scale(
    target_values("ss", nonnegative=True), force_zero_min=True
)
do_vmin, do_vmax, do_ticks, do_step = nice_scale(
    target_values("do"), force_zero_min=False
)

parameter_columns = [
    {
        "key": "turbidity",
        "title": "Turbidity",
        "unit": "NTU",
        "norm": Normalize(tur_vmin, tur_vmax, clip=True),
        "ticks": tur_ticks,
        "step": tur_step,
    },
    {
        "key": "ss",
        "title": "Suspended solids",
        "unit": "mg/L",
        "norm": Normalize(ss_vmin, ss_vmax, clip=True),
        "ticks": ss_ticks,
        "step": ss_step,
    },
    {
        "key": "do",
        "title": "DO proxy",
        "unit": "mg/L",
        "norm": Normalize(do_vmin, do_vmax, clip=True),
        "ticks": do_ticks,
        "step": do_step,
    },
]


RIVER_MAP_IDS = {"WB07", "WB05"}
LOWER_ROW_PANEL_ASPECT = 1.60  # width : height for both B and C
# Relative WB07 river interval retained after the dense coastal block is removed.
# This isolates the central meander shown in the requested close-up.
WB07_FOCUS_X_FRACTION = (0.92, 0.95)
# Keep only a narrow collar around the predicted pixels in the two lower rows.
# These values are fractions of the valid-prediction bounding box on each side.
# Increase them (for example to 0.06) if a little more satellite context is wanted.
AUTO_ZOOM_PAD_FRAC = {"WB05": 0.03, "WB07": 0.03}


def publication_support(item):
    support = np.asarray(item["display_support"], dtype=bool)
    if item["map_id"] != "WB05":
        return support
    labels, count = ndi_label(support, structure=np.ones((3, 3), dtype=np.uint8))
    if count == 0:
        return support
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == sizes.argmax()


def crop_support(item):
    """Support used only to choose the crop; displayed predictions are unchanged."""
    support = publication_support(item)
    if item["map_id"] != "WB07" or not support.any():
        return support

    # WB07 contains a broad, densely predicted coastal block at the far right.
    # Detect that block from its run of densely occupied columns and exclude most
    # of it from the crop calculation, leaving the long river as the focal area.
    n_rows, _n_cols = support.shape
    occupied_cols = np.flatnonzero(support.any(axis=0))
    if occupied_cols.size < 4:
        return support

    support_span = int(occupied_cols[-1] - occupied_cols[0] + 1)
    dense_cols = support.sum(axis=0) >= max(3, int(round(0.20 * n_rows)))
    dense_labels, dense_count = ndi_label(dense_cols.astype(np.uint8))
    candidates = []
    min_run_width = max(3, int(round(0.04 * support_span)))
    right_tolerance = max(2, int(round(0.03 * support_span)))

    for label_id in range(1, dense_count + 1):
        run_cols = np.flatnonzero(dense_labels == label_id)
        if run_cols.size < min_run_width:
            continue
        if run_cols[-1] < occupied_cols[-1] - right_tolerance:
            continue
        candidates.append(run_cols)

    if not candidates:
        return support

    coastal_run = max(candidates, key=lambda cols: cols[-1])
    # First remove the broad coastal block from the candidate crop.
    mouth_buffer = max(2, int(round(0.025 * support_span)))
    cutoff_col = min(int(coastal_run[0]) + mouth_buffer, support.shape[1])
    focused = support.copy()
    focused[:, cutoff_col:] = False
    if not focused.any():
        return support

    # Keep only the central river bend requested by the user. Fractions are
    # measured across the remaining river span, so this stays data-driven if
    # the source raster resolution changes.
    river_cols = np.flatnonzero(focused.any(axis=0))
    river_left = int(river_cols[0])
    river_span = int(river_cols[-1] - river_cols[0] + 1)
    focus_start, focus_end = WB07_FOCUS_X_FRACTION
    focus_col_lo = river_left + int(round(focus_start * river_span))
    focus_col_hi = river_left + int(round(focus_end * river_span))
    focus_col_hi = max(focus_col_hi, focus_col_lo + 1)

    focused[:, :focus_col_lo] = False
    focused[:, focus_col_hi:] = False
    return focused if focused.any() else support


def expand_bounds_to_aspect(bounds, target_aspect):
    """Expand the short dimension to a target width:height without clipping."""
    left, bottom, right, top = bounds
    width = max(right - left, 1e-9)
    height = max(top - bottom, 1e-9)
    center_x = (left + right) / 2
    center_y = (bottom + top) / 2

    if width / height > target_aspect:
        height = width / target_aspect
    else:
        width = height * target_aspect
    return (
        center_x - width / 2,
        center_y - height / 2,
        center_x + width / 2,
        center_y + height / 2,
    )


def support_mask_bounds(support, extent, origin="upper", pad_frac=0.15):
    """Return the padded map-coordinate bounding box of True support pixels."""
    rows, cols = np.where(support)
    if rows.size == 0:
        return None

    n_rows, n_cols = support.shape
    x0, x1, y0, y1 = extent
    col_lo, col_hi = cols.min(), cols.max() + 1
    row_lo, row_hi = rows.min(), rows.max() + 1

    x_left = x0 + (col_lo / n_cols) * (x1 - x0)
    x_right = x0 + (col_hi / n_cols) * (x1 - x0)

    if origin == "upper":
        y_top = y1 - (row_lo / n_rows) * (y1 - y0)
        y_bottom = y1 - (row_hi / n_rows) * (y1 - y0)
    else:
        y_bottom = y0 + (row_lo / n_rows) * (y1 - y0)
        y_top = y0 + (row_hi / n_rows) * (y1 - y0)

    width = max(x_right - x_left, 1e-9)
    height = max(y_top - y_bottom, 1e-9)
    return (
        x_left - width * pad_frac,
        y_bottom - height * pad_frac,
        x_right + width * pad_frac,
        y_top + height * pad_frac,
    )


PLOT_BOUNDS_BY_MAP_ID = {}
for item in representative_data:
    map_id = item["map_id"]
    bounds = item["bounds"]
    if map_id in RIVER_MAP_IDS:
        support = crop_support(item)
        auto_bounds = support_mask_bounds(
            support,
            item["extent"],
            origin=item["origin"],
            pad_frac=AUTO_ZOOM_PAD_FRAC.get(map_id, 0.15),
        )
        if auto_bounds is not None:
            # B and C now share the same physical panel ratio. Bounds are only
            # expanded, never contracted, so the focused prediction is not cut.
            bounds = expand_bounds_to_aspect(
                auto_bounds, target_aspect=LOWER_ROW_PANEL_ASPECT
            )
    PLOT_BOUNDS_BY_MAP_ID[map_id] = bounds


def add_north_arrow(axis):
    axis.annotate(
        "N",
        xy=(0.94, 0.91),
        xytext=(0.94, 0.79),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=7,
        fontweight="bold",
        color="#202020",
        arrowprops={"arrowstyle": "-|>", "linewidth": 0.9, "color": "#202020"},
        zorder=10,
    )


GEOD = Geod(ellps="WGS84")
SCALE_CHOICES_KM = np.array(
    [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200], dtype=float
)


def add_geodesic_scale_bar(axis, map_crs, bounds):
    left, bottom, right, top = bounds
    to_wgs84 = Transformer.from_crs(map_crs, "EPSG:4326", always_xy=True)
    from_wgs84 = Transformer.from_crs("EPSG:4326", map_crs, always_xy=True)
    y = bottom + 0.075 * (top - bottom)
    x0 = left + 0.065 * (right - left)
    lon0, lat0 = to_wgs84.transform(x0, y)
    lon_right, lat_right = to_wgs84.transform(right, y)
    _, _, width_m = GEOD.inv(lon0, lat0, lon_right, lat_right)
    target_km = max(width_m / 1000 * 0.23, SCALE_CHOICES_KM[0])
    candidates = SCALE_CHOICES_KM[SCALE_CHOICES_KM <= target_km]
    length_km = float(candidates[-1] if candidates.size else SCALE_CHOICES_KM[0])
    lon1, lat1, _ = GEOD.fwd(lon0, lat0, 90, length_km * 1000)
    x1, y1 = from_wgs84.transform(lon1, lat1)

    axis.plot(
        [x0, x1], [y, y1], color="white", linewidth=5.0,
        solid_capstyle="butt", zorder=20,
    )
    axis.plot(
        [x0, x1], [y, y1], color="#111111", linewidth=2.0,
        solid_capstyle="butt", zorder=21,
    )
    tick_height = 0.018 * (top - bottom)
    for x_tick, y_tick in [(x0, y), (x1, y1)]:
        axis.plot(
            [x_tick, x_tick],
            [y_tick - tick_height, y_tick + tick_height],
            color="#111111",
            linewidth=1.0,
            zorder=21,
        )
    axis.text(
        (x0 + x1) / 2,
        max(y, y1) + 1.7 * tick_height,
        f"{length_km:g} km",
        ha="center",
        va="bottom",
        fontsize=7,
        fontweight="bold",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 1.1},
        zorder=22,
    )


def load_thailand_boundary():
    reference_dir = HANDOFF_ROOT / "data_ready/maps/reference"
    reference_dir.mkdir(parents=True, exist_ok=True)
    boundary_zip = reference_dir / "ne_110m_admin_0_countries.zip"
    if not boundary_zip.exists():
        urlretrieve(
            "https://naturalearth.s3.amazonaws.com/110m_cultural/"
            "ne_110m_admin_0_countries.zip",
            boundary_zip,
        )
    world = gpd.read_file(f"zip://{boundary_zip.as_posix()}")
    thailand = world.loc[
        (world["ADMIN"] == "Thailand") | (world["ADM0_A3"] == "THA")
    ].to_crs("EPSG:4326")
    if thailand.empty:
        raise ValueError("Thailand was not found in the Natural Earth boundary file.")
    return thailand


def representative_context_wgs84():
    """Return each panel centre and displayed extent in longitude/latitude."""
    contexts = []
    for letter, item in zip(["A", "B", "C"], representative_data):
        date_token = item["date"].replace("-", "")
        path = GEOTIFF_DIR / f"{item['map_id']}_{date_token}_water_quality_predictions.tif"
        with rasterio.open(path) as src:
            map_crs = src.crs

        left, bottom, right, top = PLOT_BOUNDS_BY_MAP_ID[item["map_id"]]
        transformer = Transformer.from_crs(map_crs, "EPSG:4326", always_xy=True)
        corner_x = [left, left, right, right]
        corner_y = [bottom, top, bottom, top]
        corner_lon, corner_lat = transformer.transform(corner_x, corner_y)
        bbox = (
            float(np.min(corner_lon)),
            float(np.min(corner_lat)),
            float(np.max(corner_lon)),
            float(np.max(corner_lat)),
        )
        contexts.append(
            {
                "letter": letter,
                "map_id": item["map_id"],
                "lon": (bbox[0] + bbox[2]) / 2,
                "lat": (bbox[1] + bbox[3]) / 2,
                "bbox": bbox,
            }
        )
    return contexts


def shared_regional_bounds(contexts, min_width=5.4, min_height=8.0, pad_frac=0.10):
    """Build one stable southern-Thailand extent shared by all locator panels."""
    left = min(site["bbox"][0] for site in contexts)
    bottom = min(site["bbox"][1] for site in contexts)
    right = max(site["bbox"][2] for site in contexts)
    top = max(site["bbox"][3] for site in contexts)

    width = max(right - left, 1e-6)
    height = max(top - bottom, 1e-6)
    center_x = (left + right) / 2
    center_y = (bottom + top) / 2
    width = max(width * (1 + 2 * pad_frac), min_width)
    height = max(height * (1 + 2 * pad_frac), min_height)
    return (
        center_x - width / 2,
        center_y - height / 2,
        center_x + width / 2,
        center_y + height / 2,
    )


def degree_east(value, _position):
    return f"{value:g}\N{DEGREE SIGN}E"


def degree_north(value, _position):
    return f"{value:g}\N{DEGREE SIGN}N"


def draw_region_map(axis, selected_index, thailand, contexts, region_bounds):
    """Draw one row-specific regional locator in the first grid column."""
    selected = contexts[selected_index]
    left, bottom, right, top = region_bounds

    axis.set_facecolor("#DCEAF0")
    thailand.plot(
        ax=axis,
        color="#EEF0E9",
        edgecolor="#4A4A4A",
        linewidth=0.65,
        zorder=1,
    )

    # Show all representative locations for context; emphasize this row only.
    for site_index, site in enumerate(contexts):
        is_selected = site_index == selected_index
        axis.scatter(
            site["lon"],
            site["lat"],
            s=38 if is_selected else 18,
            marker="o",
            facecolor="#D73027" if is_selected else "#8B8B8B",
            edgecolor="white",
            linewidth=0.8 if is_selected else 0.5,
            alpha=1.0 if is_selected else 0.85,
            zorder=6 if is_selected else 5,
        )

    box_left, box_bottom, box_right, box_top = selected["bbox"]
    axis.add_patch(
        Rectangle(
            (box_left, box_bottom),
            box_right - box_left,
            box_top - box_bottom,
            fill=False,
            edgecolor="#111111",
            linewidth=1.0,
            linestyle="--",
            zorder=7,
        )
    )
    axis.annotate(
        f"{selected['letter']}  {selected['map_id']}",
        xy=(selected["lon"], selected["lat"]),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=7.2,
        fontweight="bold",
        ha="left",
        va="bottom",
        bbox={
            "facecolor": "white",
            "edgecolor": "#666666",
            "linewidth": 0.45,
            "alpha": 0.88,
            "pad": 1.5,
        },
        zorder=8,
    )

    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)
    axis.set_aspect("equal", adjustable="box")
    axis.xaxis.set_major_locator(MaxNLocator(3, integer=True))
    axis.yaxis.set_major_locator(MaxNLocator(5, integer=True))
    axis.xaxis.set_major_formatter(FuncFormatter(degree_east))
    axis.yaxis.set_major_formatter(FuncFormatter(degree_north))
    axis.tick_params(axis="both", labelsize=5.5, length=2.5, width=0.45, pad=1.5)
    for spine in axis.spines.values():
        spine.set_color("#606060")
        spine.set_linewidth(0.55)

    add_north_arrow(axis)
    add_geodesic_scale_bar(axis, "EPSG:4326", region_bounds)

    # Full-country inset mirrors the reference layout and locates the region.
    inset_axis = axis.inset_axes([0.055, 0.65, 0.39, 0.31])
    inset_axis.set_facecolor("white")
    thailand.plot(
        ax=inset_axis,
        color="#F1F1F1",
        edgecolor="#3D3D3D",
        linewidth=0.45,
        zorder=1,
    )
    inset_axis.scatter(
        selected["lon"],
        selected["lat"],
        s=12,
        facecolor="#D73027",
        edgecolor="white",
        linewidth=0.35,
        zorder=4,
    )
    inset_axis.set_xlim(97.0, 106.2)
    inset_axis.set_ylim(5.0, 21.0)
    inset_axis.set_aspect("equal", adjustable="box")
    inset_axis.set_xticks([])
    inset_axis.set_yticks([])
    inset_axis.set_title("Thailand", fontsize=5.8, fontweight="bold", pad=1)
    for spine in inset_axis.spines.values():
        spine.set_color("#777777")
        spine.set_linewidth(0.4)


# ---------------------------------------------------------------------------
# Four-column physical layout:
#   0 = repeated regional locator
#   1..3 = predicted parameter maps
# Dedicated header and spacer rows prevent A/B/C text from touching maps.
# ---------------------------------------------------------------------------
SITE_CONTEXTS = representative_context_wgs84()
THAILAND_BOUNDARY = load_thailand_boundary()
REGION_BOUNDS_WGS84 = shared_regional_bounds(SITE_CONTEXTS)

FIG_WIDTH_IN = 15.8
LEFT_FRAC, RIGHT_FRAC = 0.045, 0.990
TOP_FRAC, BOTTOM_FRAC = 0.992, 0.012
WSPACE_FRAC = 0.10
COLUMN_WIDTH_RATIOS = [0.72, 1.0, 1.0, 1.0]

# GridSpec defines wspace as a fraction of the average column width. This
# calculation gives the real width of one prediction panel in inches.
usable_width_in = FIG_WIDTH_IN * (RIGHT_FRAC - LEFT_FRAC)
average_cell_width_in = usable_width_in / (4 + 3 * WSPACE_FRAC)
prediction_col_width_in = (
    average_cell_width_in * 4 / sum(COLUMN_WIDTH_RATIOS)
)

# The figure title/subtitle have been removed; retain only a small top margin.
TITLE_IN = 0.10
COLUMN_TITLE_IN = 0.44
HEADER_IN = 0.48
ROW_GAP_IN = 0.16
COLORBAR_IN = 0.88

map_row_aspects = []
for item in representative_data:
    if item["map_id"] in RIVER_MAP_IDS:
        # Rows B and C use exactly the same map-panel ratio.
        map_row_aspects.append(1.0 / LOWER_ROW_PANEL_ASPECT)
    else:
        left, bottom, right, top = PLOT_BOUNDS_BY_MAP_ID[item["map_id"]]
        map_row_aspects.append(
            float(np.clip((top - bottom) / (right - left), 0.42, 1.45))
        )
map_heights_in = [ratio * prediction_col_width_in for ratio in map_row_aspects]

GRID_HEIGHT_IN = (
    TITLE_IN
    + COLUMN_TITLE_IN
    + 3 * HEADER_IN
    + sum(map_heights_in)
    + 3 * ROW_GAP_IN
    + COLORBAR_IN
)
FIG_HEIGHT_IN = GRID_HEIGHT_IN / (TOP_FRAC - BOTTOM_FRAC)

figure = plt.figure(
    figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
    facecolor="white",
    constrained_layout=False,
)
grid = GridSpec(
    12,
    4,
    figure=figure,
    width_ratios=COLUMN_WIDTH_RATIOS,
    height_ratios=[
        TITLE_IN,
        COLUMN_TITLE_IN,
        HEADER_IN,
        map_heights_in[0],
        ROW_GAP_IN,
        HEADER_IN,
        map_heights_in[1],
        ROW_GAP_IN,
        HEADER_IN,
        map_heights_in[2],
        ROW_GAP_IN,
        COLORBAR_IN,
    ],
    hspace=0.0,
    wspace=WSPACE_FRAC,
    left=LEFT_FRAC,
    right=RIGHT_FRAC,
    top=TOP_FRAC,
    bottom=BOTTOM_FRAC,
)


# Deliberately blank: no figure title or subtitle above the column names.
title_axis = figure.add_subplot(grid[0, :])
title_axis.set_axis_off()


# Column headings: regional context + three parameters.
region_title_axis = figure.add_subplot(grid[1, 0])
region_title_axis.set_axis_off()
region_title_axis.text(
    0.5,
    0.45,
    "Regional study area",
    transform=region_title_axis.transAxes,
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
)

for parameter_index, parameter in enumerate(parameter_columns):
    axis = figure.add_subplot(grid[1, parameter_index + 1])
    axis.set_axis_off()
    axis.text(
        0.5,
        0.45,
        parameter["title"],
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
    )


header_rows = [2, 5, 8]
map_rows = [3, 6, 9]
gap_rows = [4, 7, 10]
letters = ["A", "B", "C"]

for row_index, item in enumerate(representative_data):
    # Opaque, dedicated header band keeps long row titles out of map panels.
    header_axis = figure.add_subplot(grid[header_rows[row_index], :])
    header_axis.set_axis_off()
    header_axis.add_patch(
        Rectangle(
            (0, 0),
            1,
            1,
            transform=header_axis.transAxes,
            facecolor="white",
            edgecolor="none",
            zorder=0,
        )
    )
    header_axis.text(
        0.0,
        0.58,
        f"{letters[row_index]}. {item['name']} ({item['map_id']}; {item['date']})",
        transform=header_axis.transAxes,
        ha="left",
        va="center",
        fontsize=10.2,
        fontweight="bold",
        clip_on=False,
        zorder=2,
    )
    header_axis.plot(
        [0, 1],
        [0.06, 0.06],
        transform=header_axis.transAxes,
        color="#A0A0A0",
        linewidth=0.5,
        zorder=2,
    )

    # First column: a row-specific regional map.
    regional_axis = figure.add_subplot(grid[map_rows[row_index], 0])
    draw_region_map(
        regional_axis,
        selected_index=row_index,
        thailand=THAILAND_BOUNDARY,
        contexts=SITE_CONTEXTS,
        region_bounds=REGION_BOUNDS_WGS84,
    )

    map_id = item["map_id"]
    date_token = item["date"].replace("-", "")
    prediction_path = GEOTIFF_DIR / f"{map_id}_{date_token}_water_quality_predictions.tif"
    with rasterio.open(prediction_path) as src:
        map_crs = src.crs

    plot_bounds = PLOT_BOUNDS_BY_MAP_ID[map_id]
    extent = item["extent"]
    origin = item["origin"]
    support = publication_support(item)

    # Prediction maps move one column to the right (columns 1, 2, 3).
    for parameter_index, parameter in enumerate(parameter_columns):
        axis = figure.add_subplot(grid[map_rows[row_index], parameter_index + 1])
        add_light_satellite_background(
            axis=axis,
            crs=map_crs,
            bounds=plot_bounds,
            fallback_rgb=item["rgb"],
            fallback_valid=item["rgb_valid"],
            extent=extent,
            origin=origin,
        )

        values = np.asarray(item["arrays"][parameter["key"]], dtype=float).copy()
        values[~support] = np.nan
        if parameter["key"] in {"turbidity", "ss"}:
            values[values < 0] = np.nan

        axis.imshow(
            np.ma.masked_invalid(values),
            extent=extent,
            origin=origin,
            cmap=UNIFIED_CMAP,
            norm=parameter["norm"],
            alpha=0.90,
            interpolation="nearest",
            zorder=4,
        )
        if support.any() and (~support).any():
            axis.contour(
                support.astype(float),
                levels=[0.5],
                extent=extent,
                origin=origin,
                colors=["#303030"],
                linewidths=0.30,
                zorder=5,
            )

        axis.set_xlim(plot_bounds[0], plot_bounds[2])
        axis.set_ylim(plot_bounds[1], plot_bounds[3])
        axis.set_aspect("equal", adjustable="box")
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_color("#606060")
            spine.set_linewidth(0.45)

        if parameter_index == 0:
            add_geodesic_scale_bar(axis, map_crs, plot_bounds)
        if parameter_index == 2:
            add_north_arrow(axis)

    # Explicit blank separator after each map row.
    gap_axis = figure.add_subplot(grid[gap_rows[row_index], :])
    gap_axis.set_axis_off()


# A compact regional-map legend occupies the bottom-left cell.
region_legend_axis = figure.add_subplot(grid[11, 0])
region_legend_axis.set_axis_off()
region_legend_axis.scatter(
    [0.08], [0.63], s=28, facecolor="#D73027", edgecolor="white", linewidth=0.7,
    transform=region_legend_axis.transAxes, clip_on=False,
)
region_legend_axis.text(
    0.16, 0.63, "Selected representative site", transform=region_legend_axis.transAxes,
    ha="left", va="center", fontsize=7.2,
)
region_legend_axis.add_patch(
    Rectangle(
        (0.055, 0.19), 0.06, 0.17, transform=region_legend_axis.transAxes,
        fill=False, edgecolor="#111111", linewidth=0.9, linestyle="--",
    )
)
region_legend_axis.text(
    0.16, 0.275, "Prediction-panel extent", transform=region_legend_axis.transAxes,
    ha="left", va="center", fontsize=7.2,
)


# Shared parameter scales remain at the bottom, aligned under columns 1..3.
for parameter_index, parameter in enumerate(parameter_columns):
    color_grid = grid[11, parameter_index + 1].subgridspec(
        1, 3, width_ratios=[0.08, 0.84, 0.08], wspace=0
    )
    color_axis = figure.add_subplot(color_grid[0, 1])
    mappable = ScalarMappable(norm=parameter["norm"], cmap=UNIFIED_CMAP)
    mappable.set_array([])
    colorbar = figure.colorbar(
        mappable,
        cax=color_axis,
        orientation="horizontal",
        ticks=parameter["ticks"],
    )
    colorbar.ax.set_xticklabels(format_ticks(parameter["ticks"], parameter["step"]))
    colorbar.ax.tick_params(axis="x", labelsize=7, length=3, width=0.6, pad=3)
    labels = colorbar.ax.get_xticklabels()
    if labels:
        labels[0].set_ha("left")
        labels[-1].set_ha("right")
    colorbar.outline.set_linewidth(0.6)
    colorbar.set_label(
        f"Predicted {parameter['title']} ({parameter['unit']})",
        fontsize=8,
        fontweight="bold",
        labelpad=5,
    )


final_png = FINAL_FIGURE_DIR / "representative_predictions_4col_region.png"
final_pdf = FINAL_FIGURE_DIR / "representative_predictions_4col_region.pdf"
final_tiff = FINAL_FIGURE_DIR / "representative_predictions_4col_region.tif"

# Do not use bbox_inches="tight" here: the layout has already been computed in
# physical inches, and a second automatic crop can shift the header baselines.
figure.savefig(final_png, dpi=600, facecolor="white")
figure.savefig(final_pdf, facecolor="white")
figure.savefig(final_tiff, dpi=400, facecolor="white")
plt.show()
plt.close(figure)

print("PNG :", final_png)
print("PDF :", final_pdf)
print("TIFF:", final_tiff)


In [ ]:
# Optional: choose your own private destination and uncomment to persist results.
# import shutil
# destination = Path("/content/drive/MyDrive/wqr_my_results")
# shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)
print("Results for this run:", OUTPUT_ROOT)
